# Metrics Tables

Builds two inter-rater agreement tables (your annotations vs. model annotations):

- **Table 1** — 13 CRI permutations × 3 models, metrics macro-averaged across 18 features
- **Table 2** — 18 features × 3 models, full CRI only

Metrics: Precision, Recall, F1, Cohen's Kappa, Krippendorff's Alpha (nominal)

In [1]:
# 1. Imports and paths

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import precision_score, recall_score, f1_score, cohen_kappa_score

SOURCES_DIR = Path("/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/Comps/Results")
OUT_DIR     = SOURCES_DIR.parent

MODELS = ["Anthropic"] #, "Gemini", "OpenAI"
PERMS  = (
    ["full"] #+
    #[f"abl{i}" for i in range(1, 7)]
    # + [f"all{i}" for i in range(1, 7)]
)
METRICS = ["Precision", "Recall", "F1", "Kappa", "Alpha"]

In [2]:
# 2. Detect the 18 features from a sample file

sample = pd.read_csv(SOURCES_DIR / "Anthropic_full_comp.csv")
FEATURES = [c[:-3] for c in sample.columns if c.endswith("_KD")]
print(f"{len(FEATURES)} features: {FEATURES}")

16 features: ['ASTrns', 'ASUncc', 'ASUnrg', 'AktAcc', 'AktAch', 'AktAct', 'AktState', 'ClCntfcl', 'ClNegPol', 'DtExp', 'DtRes', 'DtUniv', 'TLAspPfv', 'TLMdIrr', 'TLTS', 'VcPass']


In [3]:
# 3. Metric helpers

def get_feat_cols(df, feat):
    """
    Auto-detects the KD and model columns for a feature, regardless of
    naming convention (e.g. {feat}_KD vs {feat}_KD_annotations_350).
    """
    candidates = [c for c in df.columns if c.startswith(feat + "_")]
    kd_col    = next(c for c in candidates if "_KD" in c)
    model_col = next(c for c in candidates if "_KD" not in c)
    return kd_col, model_col


def krippendorff_alpha_nominal(y1, y2):
    """
    Krippendorff's alpha for two annotators, nominal binary data.
    Derived analytically from the coincidence-matrix formulation.
    """
    y1, y2 = np.array(y1, dtype=float), np.array(y2, dtype=float)
    n = len(y1)
    if n < 2:
        return np.nan
    n_disagree = np.sum(y1 != y2)
    n0 = np.sum(y1 == 0) + np.sum(y2 == 0)
    n1 = np.sum(y1 == 1) + np.sum(y2 == 1)
    n_total = 2 * n
    D_o = n_disagree / n
    D_e = 2 * n0 * n1 / (n_total * (n_total - 1))
    if D_e == 0:
        return 1.0 if D_o == 0 else np.nan
    return 1.0 - D_o / D_e


def compute_metrics(y_true, y_pred):
    """
    Computes all five metrics for one (annotator, model) column pair.
    Drops NaN rows before computing. Returns NaN for any metric that
    cannot be computed (e.g. constant column for kappa).
    """
    s = pd.DataFrame({"t": y_true, "p": y_pred}).dropna()
    yt = s["t"].astype(int).values
    yp = s["p"].astype(int).values

    precision = precision_score(yt, yp, zero_division=0)
    recall    = recall_score(yt, yp, zero_division=0)
    f1        = f1_score(yt, yp, zero_division=0)

    try:
        kappa = cohen_kappa_score(yt, yp)
    except ValueError:
        kappa = np.nan

    alpha = krippendorff_alpha_nominal(yt, yp)

    return {"Precision": precision, "Recall": recall, "F1": f1,
            "Kappa": kappa, "Alpha": alpha}

In [4]:
# 4. Table 1 — macro-averaged metrics per CRI permutation

rows1 = []
for perm in PERMS:
    row = {"Permutation": perm}
    for model in MODELS:
        df = pd.read_csv(SOURCES_DIR / f"{model}_{perm}_comp.csv")
        feature_metrics = []
        for feat in FEATURES:
            kd_col, model_col = get_feat_cols(df, feat)
            m = compute_metrics(df[kd_col], df[model_col])
            feature_metrics.append(m)
        for metric in METRICS:
            vals = [fm[metric] for fm in feature_metrics]
            row[f"{model}_{metric}"] = np.nanmean(vals)
    rows1.append(row)

table1 = pd.DataFrame(rows1).set_index("Permutation")
flat_cols = [f"{m}_{met}" for m in MODELS for met in METRICS]
table1 = table1[flat_cols]
table1.columns = pd.MultiIndex.from_product([MODELS, METRICS], names=["Model", "Metric"])

print("Table 1: Agreement metrics by CRI permutation (macro-averaged across 18 features)")
display(table1.round(3))

Table 1: Agreement metrics by CRI permutation (macro-averaged across 18 features)


Model       Anthropic                           
Metric      Precision Recall     F1  Kappa Alpha
Permutation                                     
full            0.773  0.734  0.733  0.624  0.62

In [5]:
# 5. Table 2 — per-feature metrics, full CRI only

rows2 = []
for feat in FEATURES:
    row = {"Feature": feat}
    for model in MODELS:
        df = pd.read_csv(SOURCES_DIR / f"{model}_full_comp.csv")
        kd_col, model_col = get_feat_cols(df, feat)
        m = compute_metrics(df[kd_col], df[model_col])
        for metric in METRICS:
            row[f"{model}_{metric}"] = m[metric]
    rows2.append(row)

table2 = pd.DataFrame(rows2).set_index("Feature")
table2 = table2[flat_cols]
table2.columns = pd.MultiIndex.from_product([MODELS, METRICS], names=["Model", "Metric"])

print("Table 2: Agreement metrics by feature, full CRI only")
display(table2.round(3))

Table 2: Agreement metrics by feature, full CRI only


Model    Anthropic                            
Metric   Precision Recall     F1  Kappa  Alpha
Feature                                       
ASTrns       0.933  0.857  0.893  0.671  0.670
ASUncc       0.920  0.802  0.857  0.814  0.814
ASUnrg       0.429  0.353  0.387  0.359  0.359
AktAcc       0.504  0.735  0.598  0.439  0.432
AktAch       0.736  0.893  0.807  0.528  0.522
AktAct       0.692  0.250  0.367  0.330  0.320
AktState     0.897  0.605  0.722  0.691  0.691
ClCntfcl     0.844  0.964  0.900  0.891  0.891
ClNegPol     0.824  0.977  0.894  0.877  0.877
DtExp        0.705  0.717  0.711  0.556  0.557
DtRes        0.868  0.655  0.747  0.560  0.554
DtUniv       0.792  0.333  0.469  0.412  0.400
TLAspPfv     0.959  0.810  0.879  0.482  0.470
TLMdIrr      0.800  0.909  0.851  0.828  0.828
TLTS         0.634  0.959  0.763  0.681  0.677
VcPass       0.837  0.923  0.878  0.862  0.862

In [6]:
# 6. Export

table1.to_csv(OUT_DIR / "table1_by_permutation.csv")
table2.to_csv(OUT_DIR / "table2_by_feature.csv")
print(f"Saved to {OUT_DIR}:")
print("  table1_by_permutation.csv")
print("  table2_by_feature.csv")

Saved to /Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/Comps:
  table1_by_permutation.csv
  table2_by_feature.csv
